In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka \
    great-expectations \
    altair==4.2.2 \
    redis

In [0]:
# Cell 1 - 환경설정
import os, sys, json, uuid, io
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()
vault.get_storage_client("datacopsadls")

ACCOUNT      = "datacopsadls"
BASE_BRONZE  = f"abfss://bronze@{ACCOUNT}.dfs.core.windows.net"

print("[OK] 환경설정 완료")

In [0]:
KAFKA_BOOTSTRAP = vault.get_secret("kafka-bootstrap-servers")
KAFKA_USERNAME  = vault.get_secret("kafka-username")
KAFKA_PASSWORD  = vault.get_secret("kafka-password")

TOPIC_PATTERN = ".*\\.(batch|stream)\\.raw"

df_kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("kafka.security.protocol", "SASL_PLAINTEXT")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
            f'username="{KAFKA_USERNAME}" password="{KAFKA_PASSWORD}";')
    .option("subscribePattern", TOPIC_PATTERN)
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

print("[OK] Kafka readStream 정의 완료")
print(f"  구독 패턴: {TOPIC_PATTERN}")

In [0]:
def parse_topic(topic: str):
    """
    'samsung.ecommerce.batch.raw' → ('samsung', 'ecommerce', 'batch')
    반환: (bronze_folder, source_type)
    """
    parts = topic.split(".")
    if len(parts) < 3:
        return None, None
    company     = parts[0]
    domain      = parts[1]
    source_type = parts[2]  # batch 또는 stream
    return f"{company}_{domain}", source_type

parse_topic_udf = F.udf(lambda t: parse_topic(t)[0], StringType())
source_type_udf = F.udf(lambda t: parse_topic(t)[1], StringType())

df_parsed = (
    df_kafka_raw
    .withColumn("bronze_folder", parse_topic_udf(F.col("topic")))
    .withColumn("source_type",   source_type_udf(F.col("topic")))
    .withColumn("kafka_value",   F.col("value").cast("string"))
    .withColumn("kafka_offset",  F.col("offset"))
    .withColumn("kafka_ts",      F.col("timestamp"))
    .filter(F.col("bronze_folder").isNotNull())
)

print("[OK] 토픽 파싱 UDF 정의 완료")

In [0]:
from delta.tables import DeltaTable

def write_to_bronze(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    topics = (
        batch_df.select("topic", "bronze_folder", "source_type")
        .distinct()
        .collect()
    )

    for row in topics:
        topic         = row["topic"]
        bronze_folder = row["bronze_folder"]
        source_type   = row["source_type"]

        df_topic = batch_df.filter(F.col("topic") == topic)

        df_to_save = (
            df_topic
            .select(
                F.col("kafka_value").alias("raw_data"),
                F.col("kafka_ts").alias("_kafka_ts"),
                F.col("kafka_offset").alias("_kafka_offset"),
                F.col("source_type").alias("_source_type"),
                F.col("topic").alias("_topic"),
            )
            .withColumn("_ingest_ts", F.current_timestamp())
        )

        out_path = f"{BASE_BRONZE}/{bronze_folder}"

        df_to_save.write \
            .format("delta") \
            .mode("append") \
            .save(out_path)

        cnt = df_to_save.count()
        print(f"  [OK] {topic} → {bronze_folder}/ | {cnt}행 | batch_id={batch_id}")


query = (
    df_parsed.writeStream
    .foreachBatch(write_to_bronze)
    .option("checkpointLocation", f"{BASE_BRONZE}/_checkpoints/kafka_ingestor")
    .trigger(processingTime="30 seconds")
    .start()
)

print("[OK] Kafka → Bronze 스트리밍 시작")
print(f"  체크포인트: {BASE_BRONZE}/_checkpoints/kafka_ingestor")
query.awaitTermination()